# LRU and DuckDB

In [ ]:
# simple LRU practice: three books on a desk

# load packages
from collections import OrderedDict

class TestLRUCache:

    def __init__(self, capacity: int):
        self.cache = OrderedDict()
        self.capacity = capacity

    def get(self, key: str) -> str | None:
        """
        Get data from cache (stored in dictionary format: key-value pairs)

        Args:
            key: The key to retrieve

        Returns:
            - If key is found: return the corresponding value and move the pair
            to the end of cache (mark as most recently used)
            - Otherwise: return None
        """
        if key not in self.cache:
            return None
        value = self.cache.pop(key)
        self.cache[key] = value
        return value

    def put(self, key: str, value: str) -> None:
        """
        Create or update data in cache

        Two scenarios:
        1. If the pair already exists in the cache:
            - Pop it out and put it back again (move to end, mark as most recently used)

        2. If the cache is full:
            - Remove the oldest pair using popitem(last=False) [LRU eviction]
            - Then add the new pair
        """
        if key in self.cache:
            # OrderedDict method pop
            # It pops the existing key out, it does not matter where the position's of existing key
            self.cache.pop(key)
        elif len(self.cache) >= self.capacity:
            # OrderedDict method popitem, differ from dict's popitem
            # It removes the first (last=False) or last (last=True) element of a dict. 
            self.cache.popitem(last=False)
        
        self.cache[key] = value

    def __str__(self):
        return str(self.cache)

In [ ]:
# use the above example
cache = TestLRUCache(capacity=3)

print(f"Initialize cache: {cache}")

cache.put("A", "Book A")
cache.put("B", "Book B")
cache.put("C", "Book C")

print(f"After putting three books: {cache}")

# get book A
print(f"Take Book A: {cache.get('A')}")
print(f"After taking Book A: {cache}")

# put Book D, and drop Book B
print(f"Put Book D: {cache.put('D', 'Book D')}")
print(f"After putting Book D: {cache}")

# Get Book B now
print(f"Try to get Book B: {cache.get('B')}")
print(f"Current cache: {cache}")

## Example 2: Real Project Implementation - DuckDB + LRU Cache

Simulate dashboard user behavior:
- Users query traffic data for different routes
- Use LRU Cache to cache query results
- Quantify cache performance improvements

In [ ]:
# Step 1: Setup - Load packages and connect to DuckDB
import duckdb
import pandas as pd
import time
import hashlib
from collections import OrderedDict
from pathlib import Path

# Connect to DuckDB (in-memory)
db = duckdb.connect()

# Load Parquet files from data/processed/
processed_dir = Path("../data/processed")
parquet_files = list(processed_dir.glob("*_station_hour_processed.parquet"))

print(f"found {len(parquet_files)} Parquet files：")
for f in parquet_files: # show all parquet files
    print(f"  - {f.name}")
    

# Create a view in DuckDB from all Parquet files
# the name of view is traffic_data
if parquet_files:
    parquet_pattern = str(processed_dir / "*_station_hour_processed.parquet")
    db.execute(f"""
        CREATE OR REPLACE VIEW traffic_data AS
        SELECT * FROM read_parquet('{parquet_pattern}')
    """)
    
    # Check total rows
    total_rows = db.execute("SELECT COUNT(*) FROM traffic_data").fetchone()[0]
    print(f"\n Load all parquet files to DuckDB, total rows：{total_rows:,}")
else:
    print("\n Cannot find parquet files.")

In [ ]:
# Step 2: Build a QueryCache class with LRU + TTL
class QueryCache:
    """
    LRU Cache for DuckDB query results
    - LRU eviction policy (Least Recently Used)
    - TTL support (Time-to-Live)
    - Performance metrics tracking
    """
    
    def __init__(self, max_size=3, ttl=300):
        """
        Args:
            max_size: Maximum number of cached queries (LRU limit)
            ttl: Time-to-Live in seconds (cache expiration time)
        """
        self.cache = OrderedDict()
        self.max_size = max_size
        self.ttl = ttl
        self.timestamps = {}
        
        # Metrics
        self.hit_count = 0
        self.miss_count = 0
        
    def _generate_key(self, sql, params):
        """Generate cache key from SQL + params"""
        content = f"{sql}:{params}"
        return hashlib.md5(content.encode()).hexdigest()
    
    def get(self, sql, params):
        """
        Get cached query result
        Returns None if cache miss or expired
        """
        key = self._generate_key(sql, params)
        
        # Check if key exists
        if key not in self.cache:
            self.miss_count += 1
            return None
        
        # Check if expired (TTL)
        age = time.time() - self.timestamps[key]
        if age > self.ttl:
            print(f"Cache already expired({age:.1f}s > {self.ttl}s), it was already removed.")
            del self.cache[key]
            del self.timestamps[key]
            self.miss_count += 1
            return None
        
        # Cache hit - move to end (mark as recently used)
        self.cache.move_to_end(key)
        self.hit_count += 1
        return self.cache[key]
    
    def set(self, sql, params, result):
        """Store query result in cache"""
        key = self._generate_key(sql, params)
        
        # LRU eviction: remove oldest if cache is full
        if len(self.cache) >= self.max_size and key not in self.cache:
            oldest_key, _ = self.cache.popitem(last=False)
            oldest_sql = list(self.cache.values())[0]['sql'][:50]
            print(f" Cache is full ({self.max_size} queries), removing the oldest one: {oldest_sql}...")
            del self.timestamps[oldest_key]
        
        # Store result with metadata
        self.cache[key] = {
            'sql': sql,
            'params': params,
            'result': result,
            'rows': len(result)
        }
        self.timestamps[key] = time.time()
    
    def get_stats(self):
        """Get cache statistics"""
        total = self.hit_count + self.miss_count
        hit_rate = self.hit_count / total if total > 0 else 0
        
        return {
            'cache_size': len(self.cache),
            'max_size': self.max_size,
            'total_queries': total,
            'cache_hits': self.hit_count,
            'cache_misses': self.miss_count,
            'hit_rate': hit_rate
        }
    
    def show_cache_contents(self):
        """Display current cache contents"""
        print(f"\n Cache contents:({len(self.cache)}/{self.max_size})：")
        for i, (key, value) in enumerate(self.cache.items(), 1):
            age = time.time() - self.timestamps[key]
            print(f"key: {key}")
            display(value['result'].head(5))
            print(f"  {i}. {value['sql'][:60]}... ({value['rows']} rows, {age:.1f}s ago)")
            print(f"="*70)

# Initialize cache
cache = QueryCache(max_size=3, ttl=300)
print("QueryCache Initialized(max_size=3, ttl=300 secs)")

In [ ]:
# Step 3: Define a query function with cache support
def query_traffic_by_route(route, year=None, use_cache=True):
    """
    Query traffic data by route (with optional caching)
    
    Args:
        route: Route number (e.g., 5 for I-5, 405 for I-405)
        year: Year filter (optional)
        use_cache: Whether to use cache (default: True)
    
    Returns:
        pandas DataFrame with query results
    """
    # Build SQL query
    sql = "SELECT * FROM traffic_data WHERE route = ?"
    params = [route]
    
    if year:
        sql += " AND year = ?"
        params.append(year)
    
    params_tuple = tuple(params)  # Convert to tuple for hashing
    
    # Try to get from cache first
    if use_cache:
        cached = cache.get(sql, params_tuple)
        if cached is not None:
            print(f" Cache Hit! The result from cache has:{cached['rows']} rows.")
            return cached['result']
    
    # Cache miss - execute query
    print(f"  Cache Miss, running DuckDB query...")
    start = time.time()
    result = db.execute(sql, params).df()
    duration = time.time() - start
    
    print(f"  Time spent: {duration*1000:.1f}ms({len(result)} rows)")
    
    # Store in cache
    if use_cache:
        cache.set(sql, params_tuple, result)
        print(f" Save to cache")
    
    return result

### Scenario Simulation: Dashboard User Behavior

Simulate a user's operation flow on a Streamlit Dashboard:
1. View I-5 route 2024 data (first query)
2. Switch chart type (same data, test cache)
3. View I-405 route (second route)
4. View SR-91 route (third route)
5. View I-10 route (fourth route, cache full, triggers LRU eviction)
6. Switch back to I-405 route (test LRU retains popular queries)
7. Switch back to I-5 route (evicted, needs re-query)

In [ ]:
print("=" * 70)
print("🎬 Scenario Simulation Start: Dashboard User Behavior")
print("=" * 70)

# Query 1: I-5, 2024 (First time - Cache Miss)
print("\n1️⃣ User selects I-5 route, 2024")
df1 = query_traffic_by_route(route=5, year=2024)
cache.show_cache_contents()

# Query 2: I-5, 2024 again (User switches chart type - Cache Hit!)
print("\n2️⃣ User switches chart type (same data)")
df2 = query_traffic_by_route(route=5, year=2024)
cache.show_cache_contents()

# Query 3: I-405, 2024 (Second route - Cache Miss)
print("\n3️⃣ User switches to I-405 route")
df3 = query_traffic_by_route(route=405, year=2024)
cache.show_cache_contents()

# Query 4: SR-91, 2024 (Third route - Cache Miss, cache is full now)
print("\n4️⃣ User switches to SR-91 route")
df4 = query_traffic_by_route(route=91, year=2024)
cache.show_cache_contents()

# Query 5: SR-22, 2024 (Fourth route - Cache Miss, LRU eviction!)
print("\n5️⃣ User switches to SR-22 route (cache full, triggers LRU eviction)")
df5 = query_traffic_by_route(route=22, year=2024)
cache.show_cache_contents()

# Query 6: I-405, 2024 again (Cache Hit - still in cache!)
print("\n6️⃣ User switches back to I-405 route (test LRU retention)")
df6 = query_traffic_by_route(route=405, year=2024)
cache.show_cache_contents()

# Query 7: I-5, 2024 again (Cache Miss - was evicted!)
print("\n7️⃣ User switches back to I-5 route (was evicted, needs re-query)")
df7 = query_traffic_by_route(route=5, year=2024)
cache.show_cache_contents()

print("\n" + "=" * 70)

In [ ]:
# Display final cache statistics
print("\n📊 Final Cache Statistics:")
stats = cache.get_stats()

print(f"  Total queries: {stats['total_queries']}")
print(f"  Cache hits: {stats['cache_hits']}")
print(f"  Cache misses: {stats['cache_misses']}")
print(f"  Cache hit rate: {stats['hit_rate']:.1%}")
print(f"  Current cache size: {stats['cache_size']}/{stats['max_size']}")

print("\n💡 Key Observations:")
print("  1. Query 2 for I-5 (chart switch) → Cache Hit, extremely fast!")
print("  2. Query 5 for I-10 (cache full) → Triggers LRU eviction, removes least recently used I-5")
print("  3. Query 6 for I-405 → Cache Hit, because I-405 is more recent in LRU")
print("  4. Query 7 for I-5 → Cache Miss, because I-5 was evicted")
print("  5. Hit rate = 2/7 = 28.6% (higher in real scenarios, as users often repeat queries)")

### Performance Comparison: With Cache vs Without Cache

Test performance differences for the same query repeated 10 times

In [ ]:
print("=" * 70)
print("⚡ Performance Comparison Test: Repeat I-5 Route Query 10 Times")
print("=" * 70)

# Reset cache for clean test
cache = QueryCache(max_size=10, ttl=300)

# Test 1: Without cache (disable caching)
print("\n❌ Test 1: Without Cache (query DuckDB every time)")
times_no_cache = []

for i in range(10):
    start = time.time()
    df = query_traffic_by_route(route=5, year=2024, use_cache=False)
    elapsed = time.time() - start
    times_no_cache.append(elapsed)
    if i < 3:  # Only show first 3 to save space
        print(f"  Query {i+1}: {elapsed*1000:.1f}ms")

total_no_cache = sum(times_no_cache)
print(f"  ...\n  Total time: {total_no_cache*1000:.1f}ms")

# Test 2: With cache
print("\n✅ Test 2: With Cache (first Miss, subsequent Hits)")
cache = QueryCache(max_size=10, ttl=300)  # Reset cache
times_with_cache = []

for i in range(10):
    start = time.time()
    df = query_traffic_by_route(route=5, year=2024, use_cache=True)
    elapsed = time.time() - start
    times_with_cache.append(elapsed)
    if i < 3 or i == 9:  # Show first 3 and last one
        print(f"  Query {i+1}: {elapsed*1000:.1f}ms")

total_with_cache = sum(times_with_cache)
print(f"  Total time: {total_with_cache*1000:.1f}ms")

# Calculate speedup
speedup = total_no_cache / total_with_cache
print(f"\n🚀 Performance Improvement:")
print(f"  Without cache total time: {total_no_cache*1000:.1f}ms")
print(f"  With cache total time: {total_with_cache*1000:.1f}ms")
print(f"  Speedup: {speedup:.1f}x")
print(f"  Time saved: {(total_no_cache - total_with_cache)*1000:.1f}ms")

# Cache statistics
stats = cache.get_stats()
print(f"\n📊 Cache Statistics:")
print(f"  Hit rate: {stats['hit_rate']:.1%}")